In [ ]:
QUICK = False
import os
os.makedirs("/kaggle/working/dr-failure-modes/drfm", exist_ok=True)

In [ ]:
open("/kaggle/working/dr-failure-modes/drfm/__init__.py", "w").close()

In [ ]:
%%writefile /kaggle/working/dr-failure-modes/drfm/analyze.py
import os
import glob
import json
import zlib
from multiprocessing import Pool

import cv2
import numpy as np
import pandas as pd
from scipy import stats as sst
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from . import config as C
from . import data as D
from . import metrics as M
from .corruptions import corrupt, conditions

MAIN_METRICS = ["accuracy", "qwk", "macro_f1", "referable_sens", "referable_spec", "ref_auc", "ece", "msge"]
DELTA_METRICS = ["accuracy", "qwk", "referable_sens", "referable_spec", "ref_auc", "ece", "msge"]
COLORS = {"gaussian_blur": "#1f77b4", "brightness": "#ff7f0e", "darkening": "#8c564b", "contrast": "#2ca02c",
          "gaussian_noise": "#9467bd", "jpeg": "#d62728", "compound_blur_dark_jpeg": "#17becf",
          "compound_noise_contrast_jpeg": "#bcbd22", "compound_field": "#7f7f7f"}
NICE = {"gaussian_blur": "Gaussian blur", "brightness": "Brightness increase", "darkening": "Underexposure",
        "contrast": "Contrast reduction", "gaussian_noise": "Gaussian noise", "jpeg": "JPEG compression",
        "compound_blur_dark_jpeg": "Blur + underexposure + JPEG", "compound_noise_contrast_jpeg": "Noise + low contrast + JPEG",
        "compound_field": "Random field profile", "clean": "Clean"}
LR = {}
_W = {}
NONFINITE = {}


def T(name):
    return os.path.join(C.TABLES, name)


def F(name):
    return os.path.join(C.FIGS, name)


def save_fig(fig, name):
    fig.savefig(F(name + ".png"), dpi=300, bbox_inches="tight")
    fig.savefig(F(name + ".pdf"), bbox_inches="tight")
    plt.close(fig)


def lr_of(protocol, arch):
    return LR[arch] if protocol == "main" else C.PROTOCOLS["v1"]["lr"]


def runs(protocol, arch, domain, seeds=None):
    out = {}
    for s in seeds or C.SEEDS:
        p = os.path.join(C.RUNS, protocol, arch, domain, f"lr{lr_of(protocol, arch):g}", f"seed{s}", "logits.npz")
        if os.path.exists(p):
            with np.load(p) as z:
                r = {k: z[k] for k in z.files}
            bad = {k: int((~np.isfinite(v).all(-1)).sum()) for k, v in r.items() if v.dtype.kind == "f"}
            bad = {k: v for k, v in bad.items() if v}
            if bad:
                NONFINITE[p] = {"protocol": protocol, "arch": arch, "domain": domain, "seed": s,
                                "n_keys": len(bad), "n_rows": sum(bad.values()), "keys": ";".join(sorted(bad)[:20])}
                continue
            out[s] = r
    return out


def probs(rs, key, t=None):
    return np.stack([M.softmax(r[key], 1.0 if t is None else t[s]) for s, r in rs.items()])


def W(name, n):
    if name not in _W:
        _W[name] = M.boot_weights(n, seed=zlib.crc32(name.encode()))
    return _W[name]


def S(m, arr):
    return M.summarize(arr, basic=m.startswith("ece"))


def fmt(d, nd=3):
    return f"{d['mean']:.{nd}f} ({d['lo']:.{nd}f}-{d['hi']:.{nd}f})"


def temps(rs):
    return {s: M.fit_temperature(r["val/clean/0"], r["y/val"]) for s, r in rs.items()}


def quality_tables(meta):
    cols = ["lap_var", "tenengrad", "brenner", "brightness", "rms_contrast", "fov", "native_w", "native_h"]
    rows = []
    for split, a, b in [("median", "high", "low"), ("tertile", "high", "low")]:
        col = "quality" if split == "median" else "tertile"
        A, B = meta[meta[col] == a], meta[meta[col] == b]
        for c in cols:
            p = sst.mannwhitneyu(A[c], B[c]).pvalue
            rows.append({"split": split, "feature": c, "high_median": A[c].median(), "high_q1": A[c].quantile(.25),
                         "high_q3": A[c].quantile(.75), "low_median": B[c].median(), "low_q1": B[c].quantile(.25),
                         "low_q3": B[c].quantile(.75), "mannwhitney_p": p,
                         "cliffs_delta": 2 * sst.mannwhitneyu(A[c], B[c]).statistic / (len(A) * len(B)) - 1})
    pd.DataFrame(rows).to_csv(T("table_quality_domains.csv"), index=False)
    g = []
    for split, col in [("median", "quality"), ("tertile", "tertile")]:
        ct = pd.crosstab(meta[meta[col].isin(["high", "low"])][col], meta[meta[col].isin(["high", "low"])]["diagnosis"])
        chi = sst.chi2_contingency(ct.values)
        for q in ["high", "low"]:
            r = {"split": split, "domain": q, "n": int(ct.loc[q].sum()), "chi2": chi[0], "chi2_p": chi[1]}
            for k in range(C.NUM_CLASSES):
                r[f"grade_{k}_n"] = int(ct.loc[q].get(k, 0))
                r[f"grade_{k}_pct"] = 100 * ct.loc[q].get(k, 0) / ct.loc[q].sum()
            g.append(r)
    pd.DataFrame(g).to_csv(T("table_quality_grades.csv"), index=False)
    cor = []
    for c in ["tenengrad", "brenner", "brightness", "rms_contrast", "fov", "native_w", "diagnosis"]:
        r = sst.spearmanr(meta["lap_var"], meta[c])
        cor.append({"feature": c, "spearman_rho": r.statistic if hasattr(r, "statistic") else r[0], "p": r.pvalue})
    pd.DataFrame(cor).to_csv(T("table_quality_correlations.csv"), index=False)
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
    for q, col in [("high", "#1f77b4"), ("low", "#d62728")]:
        s = meta[meta["quality"] == q]
        ax[0].hist(np.log10(s["lap_var"] + 1), bins=40, alpha=.6, color=col, label=q + " quality")
        ax[3].bar(np.arange(5) + (-.2 if q == "high" else .2), np.bincount(s["diagnosis"], minlength=5) / len(s), .4, color=col, label=q)
    ax[0].set_xlabel("log10 variance of Laplacian")
    ax[0].legend(frameon=False)
    for k, c in [(1, "brightness"), (2, "rms_contrast")]:
        ax[k].boxplot([meta[meta["quality"] == q][c] for q in ["high", "low"]])
        ax[k].set_xticks([1, 2], ["high", "low"])
        ax[k].set_title(c.replace("_", " "))
    ax[0].set_title("Sharpness")
    ax[3].set_title("Grade distribution")
    ax[3].set_xlabel("Grade")
    ax[3].legend(frameon=False)
    save_fig(fig, "fig_quality_domains")


def _sev_worker(args):
    img, c, s, key = args
    out = D.center_crop(corrupt(img, c, s, key)) if c != "clean" else D.center_crop(img)
    return D.quality_metrics(cv2.cvtColor(np.ascontiguousarray(out), cv2.COLOR_RGB2GRAY))


def severity_mapping(meta, arr, splits):
    te = splits["full"][2]
    with Pool(max(1, os.cpu_count() or 1)) as pool:
        ref = pd.DataFrame(pool.map(_sev_worker, [(np.asarray(arr[i]), "clean", 0, k) for i, k in zip(range(len(meta)), meta["id_code"])], chunksize=32))
        rows = []
        for c, s in conditions(True):
            if c == "clean":
                continue
            q = pd.DataFrame(pool.map(_sev_worker, [(np.asarray(arr[i]), c, s, meta["id_code"][i]) for i in te], chunksize=16))
            r = {"corruption": c, "severity": s, "parameter": C.PARAMS[c][s - 1] if c in C.PARAMS else ""}
            for f in ["lap_var", "brightness", "rms_contrast"]:
                med = q[f].median()
                lo, hi = ref[f].quantile([.01, .99])
                r[f"{f}_median"] = med
                r[f"{f}_clean_percentile"] = 100 * (ref[f] < med).mean()
                r[f"{f}_frac_outside_clean_1_99"] = float(((q[f] < lo) | (q[f] > hi)).mean())
            rows.append(r)
    out = pd.DataFrame(rows)
    out.to_csv(T("table_severity_mapping.csv"), index=False)
    ref.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_csv(T("table_clean_quality_reference.csv"))


def benchmark():
    rows, tests, pts = [], [], []
    for protocol in ["main", "v1"]:
        stt = {}
        for a in C.ARCHS:
            rs = runs(protocol, a, "full")
            if not rs:
                continue
            y = next(iter(rs.values()))["y/test"]
            w = W("test", len(y))
            P = probs(rs, "test/clean/0")
            st = M.stats(P, y, w)
            tt = temps(rs)
            st["ece_ts"] = M.stats(probs(rs, "test/clean/0", tt), y, w)["ece"]
            stt[a] = st
            for m in MAIN_METRICS + ["ece_ts"] + [f"recall_{k}" for k in range(5)]:
                rows.append({"protocol": protocol, "arch": a, "metric": m, **S(m, st[m])})
            am = [M.point_metrics(P[i], y)["auc_macro"] for i in range(len(P))]
            rows.append({"protocol": protocol, "arch": a, "metric": "auc_macro", "mean": float(np.mean(am)),
                         "sd": float(np.std(am, ddof=1)) if len(am) > 1 else np.nan, "lo": np.nan, "hi": np.nan})
            rows.append({"protocol": protocol, "arch": a, "metric": "temperature", "mean": float(np.mean(list(tt.values()))),
                         "sd": float(np.std(list(tt.values()), ddof=1)) if len(tt) > 1 else np.nan, "lo": np.nan, "hi": np.nan})
            if protocol == "main":
                cm = sum(np.bincount(y * 5 + p.argmax(-1), minlength=25).reshape(5, 5) for p in P)
                pts.append((a, cm))
        archs = list(stt)
        for i in range(len(archs)):
            for j in range(i + 1, len(archs)):
                for m in ["qwk", "accuracy", "ece", "referable_spec", "ref_auc"]:
                    d = M.summarize_diff(stt[archs[i]][m], stt[archs[j]][m])
                    tests.append({"protocol": protocol, "a": archs[i], "b": archs[j], "metric": m, **d,
                                  "welch_p": M.welch(stt[archs[i]][m][:, 0], stt[archs[j]][m][:, 0])})
    df = pd.DataFrame(rows)
    df.to_csv(T("table_benchmark.csv"), index=False)
    if tests:
        tdf = pd.DataFrame(tests)
        tdf["p_holm"] = np.nan
        for _, g in tdf.groupby("protocol"):
            tdf.loc[g.index, "p_holm"] = M.holm(g["p"].values)
        tdf.to_csv(T("table_benchmark_tests.csv"), index=False)
    if pts:
        fig, axes = plt.subplots(1, len(pts), figsize=(4.6 * len(pts), 4.2))
        axes = np.atleast_1d(axes)
        for ax, (a, cm) in zip(axes, pts):
            cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
            ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
            ax.set_title(C.label(a))
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
            ax.set_xticks(range(5))
            ax.set_yticks(range(5))
            for i in range(5):
                for j in range(5):
                    ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", color="white" if cmn[i, j] > .5 else "black", fontsize=8)
        save_fig(fig, "fig_confusion")
    return df


def crosscond(meta, splits):
    rows, gaps, gtests, budget = [], [], [], []
    for prefix, doms, protocols in [("", ["high", "low"], ["main", "v1"]), ("t", ["thigh", "tlow"], ["main"])]:
        for protocol in protocols:
            gs = {}
            for a in C.ARCHS:
                for d in doms:
                    rs = runs(protocol, a, d, C.TERTILE_SEEDS if prefix else None)
                    if not rs:
                        continue
                    other = [x for x in doms if x != d][0]
                    st = {}
                    for td in [d, other]:
                        y = next(iter(rs.values()))[f"y/{td}_test"]
                        st[td] = M.stats(probs(rs, f"{td}_test/clean/0"), y, W(td + "_test", len(y)))
                        for m in ["accuracy", "qwk", "referable_sens", "referable_spec", "ref_auc", "msge", "ece"]:
                            rows.append({"split": "tertile" if prefix else "median", "protocol": protocol, "arch": a,
                                         "train": d, "test": td, "setting": "in_domain" if td == d else "cross_domain",
                                         "metric": m, **S(m, st[td][m])})
                    gap = np.nanmean(st[d]["qwk"], 0) - np.nanmean(st[other]["qwk"], 0)
                    per_seed = st[d]["qwk"][:, 0] - st[other]["qwk"][:, 0]
                    gs[(a, d)] = (gap, per_seed)
                    lo, hi = np.nanpercentile(gap[1:], [2.5, 97.5])
                    gaps.append({"split": "tertile" if prefix else "median", "protocol": protocol, "arch": a, "train": d,
                                 "in_domain_qwk": float(np.nanmean(st[d]["qwk"][:, 0])),
                                 "cross_domain_qwk": float(np.nanmean(st[other]["qwk"][:, 0])),
                                 "gap": float(gap[0]), "gap_lo": lo, "gap_hi": hi, "gap_p": M.boot_p(gap),
                                 "gap_sd_seeds": float(np.std(per_seed, ddof=1)) if len(per_seed) > 1 else np.nan, "n_seeds": len(rs)})
            for d in doms:
                ar = [a for a in C.ARCHS if (a, d) in gs]
                for i in range(len(ar)):
                    for j in range(i + 1, len(ar)):
                        ga, gb = gs[(ar[i], d)], gs[(ar[j], d)]
                        dd = ga[0] - gb[0]
                        lo, hi = np.nanpercentile(dd[1:], [2.5, 97.5])
                        gtests.append({"split": "tertile" if prefix else "median", "protocol": protocol, "train": d,
                                       "a": ar[i], "b": ar[j], "gap_a": float(ga[0][0]), "gap_b": float(gb[0][0]),
                                       "diff": float(dd[0]), "lo": lo, "hi": hi, "p": M.boot_p(dd), "welch_p": M.welch(ga[1], gb[1])})
    cdf = pd.DataFrame(rows)
    cdf.to_csv(T("table_crosscondition.csv"), index=False)
    gdf = pd.DataFrame(gaps)
    gdf.to_csv(T("table_generalisation_gap.csv"), index=False)
    if gtests:
        t = pd.DataFrame(gtests)
        t["p_holm"] = np.nan
        for _, g in t.groupby(["split", "protocol"]):
            t.loc[g.index, "p_holm"] = M.holm(g["p"].values)
        t.to_csv(T("table_gap_tests.csv"), index=False)
    for a in C.ARCHS:
        for d in ["full", "high", "low"]:
            rm, rv = runs("main", a, d), runs("v1", a, d)
            if not rm or not rv:
                continue
            key = "test/clean/0" if d == "full" else f"{d}_test/clean/0"
            y = next(iter(rm.values()))["y/test" if d == "full" else f"y/{d}_test"]
            w = W("test" if d == "full" else d + "_test", len(y))
            sm, sv = M.stats(probs(rm, key), y, w), M.stats(probs(rv, key), y, w)
            for m in ["qwk", "accuracy", "ece", "msge"]:
                budget.append({"arch": a, "domain": d, "metric": m, "main": M.summarize(sm[m])["mean"],
                               "v1": M.summarize(sv[m])["mean"], **{f"diff_{k}": v for k, v in M.summarize_diff(sm[m], sv[m]).items()}})
    pd.DataFrame(budget).to_csv(T("table_training_budget.csv"), index=False)
    grade_matched(splits)
    return gdf


def grade_matched(splits):
    rng = np.random.default_rng(7)
    rows = []
    for a in C.ARCHS:
        for d in ["high", "low"]:
            rs = runs("main", a, d)
            if not rs:
                continue
            other = "low" if d == "high" else "high"
            yi, yc = next(iter(rs.values()))[f"y/{d}_test"], next(iter(rs.values()))[f"y/{other}_test"]
            pi, pc = probs(rs, f"{d}_test/clean/0").argmax(-1), probs(rs, f"{other}_test/clean/0").argmax(-1)
            vals = []
            for _ in range(C.GRADE_MATCH_REPS):
                ii, cc = [], []
                for g in range(C.NUM_CLASSES):
                    a1, a2 = np.flatnonzero(yi == g), np.flatnonzero(yc == g)
                    k = min(len(a1), len(a2))
                    if k:
                        ii += list(rng.choice(a1, k, replace=False))
                        cc += list(rng.choice(a2, k, replace=False))
                ii, cc = np.array(ii), np.array(cc)
                qi = np.mean([M.from_conf(np.bincount(yi[ii] * 5 + p[ii], minlength=25).reshape(5, 5))["qwk"] for p in pi])
                qc = np.mean([M.from_conf(np.bincount(yc[cc] * 5 + p[cc], minlength=25).reshape(5, 5))["qwk"] for p in pc])
                vals.append((qi, qc, qi - qc, len(ii)))
            v = np.array(vals)
            rows.append({"arch": a, "train": d, "in_domain_qwk": v[:, 0].mean(), "cross_domain_qwk": v[:, 1].mean(),
                         "gap": v[:, 2].mean(), "gap_lo": np.percentile(v[:, 2], 2.5), "gap_hi": np.percentile(v[:, 2], 97.5),
                         "n_per_domain": int(v[0, 3])})
    pd.DataFrame(rows).to_csv(T("table_grade_matched.csv"), index=False)


def taxonomy(dd):
    a, s, p, g = dd["ref_auc"], dd["referable_sens"], dd["referable_spec"], dd["msge"]
    loss = a["hi"] < 0 and a["mean"] <= -C.AUC_MARGIN
    shift = (abs(p["mean"]) >= C.SHIFT_MARGIN and (p["hi"] < 0 or p["lo"] > 0)) or (abs(s["mean"]) >= C.SHIFT_MARGIN and (s["hi"] < 0 or s["lo"] > 0))
    if loss and shift:
        base = "mixed"
    elif loss:
        base = "discriminability_loss"
    elif shift:
        base = "operating_point_shift"
    else:
        return "robust"
    direction = "over" if g["lo"] > 0 else ("under" if g["hi"] < 0 else "neutral")
    return f"{base}_{direction}"


def robustness():
    rows, tax, ece_rows, shift_rows, curve = [], [], [], [], {}
    for protocol in ["main", "v1"]:
        for a in C.ARCHS:
            rs = runs(protocol, a, "full")
            if not rs:
                continue
            r0 = next(iter(rs.values()))
            y = r0["y/test"]
            w = W("test", len(y))
            P0 = probs(rs, "test/clean/0")
            s0 = M.stats(P0, y, w)
            tt = temps(rs) if protocol == "main" else None
            pred0 = P0.argmax(-1)
            keys = [("working", c, s, f"test/{c}/{s}") for c, s in conditions(True)]
            keys += [("native", c, s, f"native/{c}/{s}") for c in C.NATIVE_CORRUPTIONS for s in C.SEVERITIES]
            delong_rows = []
            for proto2, c, s, k in keys:
                if k not in r0:
                    continue
                P = probs(rs, k)
                st = M.stats(P, y, w)
                sh = M.shift_stats(P.argmax(-1), pred0, w)
                base = {"protocol": protocol, "arch": a, "corruption_protocol": proto2, "corruption": c, "severity": s}
                dd = {}
                for m in MAIN_METRICS:
                    rows.append({**base, "metric": m, **S(m, st[m])})
                for m in DELTA_METRICS:
                    dd[m] = M.summarize_diff(st[m], s0[m])
                    rows.append({**base, "metric": "delta_" + m, "mean": dd[m]["mean"], "sd": np.nan, "lo": dd[m]["lo"],
                                 "hi": dd[m]["hi"], "p": dd[m]["p"]})
                for m in ["pred_shift", "up_rate", "down_rate"]:
                    rows.append({**base, "metric": m, **M.summarize(sh[m])})
                if tt is not None and proto2 == "working":
                    e2 = M.stats(probs(rs, k, tt), y, w)["ece"]
                    ece_rows.append({**base, "ece": S("ece", st["ece"])["mean"], **{f"ece_ts_{q}": v for q, v in S("ece", e2).items()}})
                if c != "clean":
                    ps = [M.delong(y >= C.REFERABLE, P0[i][:, 2:].sum(-1), P[i][:, 2:].sum(-1))[2] for i in range(len(P))]
                    delong_rows.append((len(tax), ps))
                    tax.append({**base, "label": taxonomy(dd), **{f"d_{m}": dd[m]["mean"] for m in ["ref_auc", "referable_sens", "referable_spec", "msge", "qwk"]},
                                **{f"d_{m}_lo": dd[m]["lo"] for m in ["ref_auc", "referable_sens", "referable_spec", "msge"]},
                                **{f"d_{m}_hi": dd[m]["hi"] for m in ["ref_auc", "referable_sens", "referable_spec", "msge"]},
                                "delong_p_median": float(np.median(ps))})
                if protocol == "main" and proto2 == "working":
                    curve[(a, c, s)] = {m: S(m, st[m]) for m in MAIN_METRICS}
            if delong_rows:
                mat = np.array([p for _, p in delong_rows])
                adj = np.column_stack([M.holm(mat[:, j]) for j in range(mat.shape[1])])
                for (ti, _), row in zip(delong_rows, adj):
                    tax[ti]["delong_seeds_sig_holm"] = int((row < C.ALPHA).sum())
                    tax[ti]["n_seeds"] = len(row)
    df = pd.DataFrame(rows)
    df.to_csv(T("table_robustness.csv"), index=False)
    pd.DataFrame(tax).to_csv(T("table_taxonomy.csv"), index=False)
    pd.DataFrame(ece_rows).to_csv(T("table_calibration.csv"), index=False)
    wide = []
    sel = df[(df.corruption_protocol == "working") & ((df.severity == max(C.SEVERITIES)) | (df.corruption == "clean"))]
    for (protocol, a, c), g in sel.groupby(["protocol", "arch", "corruption"], sort=False):
        r = {"protocol": protocol, "arch": a, "corruption": c}
        for m in ["accuracy", "qwk", "referable_sens", "referable_spec", "ref_auc", "ece", "msge"]:
            x = g[g.metric == m].iloc[0]
            r[m] = x["mean"]
            r[m + "_ci"] = f"{x['lo']:.3f}-{x['hi']:.3f}"
        wide.append(r)
    pd.DataFrame(wide).to_csv(T("table_robustness_sev5.csv"), index=False)
    burden = []
    for x in wide:
        g = sel[(sel.protocol == x["protocol"]) & (sel.arch == x["arch"]) & (sel.corruption == x["corruption"])]
        sp, se = g[g.metric == "referable_spec"].iloc[0], g[g.metric == "referable_sens"].iloc[0]
        burden.append({"protocol": x["protocol"], "arch": x["arch"], "corruption": x["corruption"],
                       "false_pos_per_1000": (1 - sp["mean"]) * C.PREVALENCE_NONREF, "fp_lo": (1 - sp["hi"]) * C.PREVALENCE_NONREF,
                       "fp_hi": (1 - sp["lo"]) * C.PREVALENCE_NONREF, "missed_per_1000": (1 - se["mean"]) * C.PREVALENCE_REF,
                       "missed_lo": (1 - se["hi"]) * C.PREVALENCE_REF, "missed_hi": (1 - se["lo"]) * C.PREVALENCE_REF})
    pd.DataFrame(burden).to_csv(T("table_clinical_burden.csv"), index=False)
    if curve:
        fig_robustness(curve)
    return df


def fig_robustness(curve):
    archs = [a for a in C.ARCHS if any(k[0] == a for k in curve)]
    ms = [("qwk", "QWK"), ("referable_sens", "Referable sensitivity"), ("referable_spec", "Referable specificity"),
          ("ref_auc", "Referable AUC"), ("msge", "MSGE"), ("ece", "ECE")]
    for group, cs in [("single", C.CORRUPTIONS), ("compound", C.COMPOUNDS)]:
        fig, axes = plt.subplots(len(archs), len(ms), figsize=(3.3 * len(ms), 2.8 * len(archs)), squeeze=False)
        for i, a in enumerate(archs):
            for j, (m, mn) in enumerate(ms):
                ax = axes[i, j]
                for c in cs:
                    xs = [s for s in C.SEVERITIES if (a, c, s) in curve]
                    if not xs:
                        continue
                    v = [curve[(a, c, s)][m] for s in xs]
                    ax.plot(xs, [q["mean"] for q in v], marker="o", ms=3, color=COLORS[c], label=NICE[c])
                    ax.fill_between(xs, [q["lo"] for q in v], [q["hi"] for q in v], color=COLORS[c], alpha=.15, lw=0)
                ax.axhline(curve[(a, "clean", 0)][m]["mean"], ls="--", color="grey", lw=1)
                if i == 0:
                    ax.set_title(mn)
                if j == 0:
                    ax.set_ylabel(C.label(a))
                if i == len(archs) - 1:
                    ax.set_xlabel("Severity")
                ax.grid(alpha=.25)
        h, l = axes[0, 0].get_legend_handles_labels()
        fig.legend(h, l, loc="lower center", ncol=len(cs), frameon=False, bbox_to_anchor=(.5, -.03))
        fig.tight_layout(rect=[0, .04, 1, 1])
        save_fig(fig, f"fig_robustness_{group}")


def recalibration():
    rows = []
    for a in C.ARCHS:
        rs = runs("main", a, "full")
        if not rs:
            continue
        for c, s in conditions(True):
            if c == "clean":
                continue
            per = []
            for sd, r in rs.items():
                if f"val/{c}/{s}" not in r:
                    continue
                yv, yt = r["y/val"], r["y/test"]
                pv0, pv, pt = M.softmax(r["val/clean/0"]), M.softmax(r[f"val/{c}/{s}"]), M.softmax(r[f"test/{c}/{s}"])
                target = float((pv0.argmax(1) >= C.REFERABLE)[yv >= C.REFERABLE].mean())
                d_pred = pt.argmax(1) >= C.REFERABLE
                pos = yt >= C.REFERABLE
                _, sa, pa = M.recalibrated(pv0[:, 2:].sum(1), yv, pt[:, 2:].sum(1), yt, target)
                _, sb, pb = M.recalibrated(pv[:, 2:].sum(1), yv, pt[:, 2:].sum(1), yt, target)
                p0t = M.softmax(r["test/clean/0"])[:, 2:].sum(1)
                per.append([d_pred[pos].mean(), (~d_pred[~pos]).mean(), sa, pa, sb, pb, target,
                            M.spec_at_sens(p0t, yt, target), M.spec_at_sens(pt[:, 2:].sum(1), yt, target)])
            if not per:
                continue
            v = np.array(per)
            r = {"arch": a, "corruption": c, "severity": s, "target_sens": v[:, 6].mean(),
                 "clean_spec_at_target": v[:, 7].mean(), "oracle_spec_at_target": v[:, 8].mean(),
                 "oracle_spec_at_target_sd": v[:, 8].std(ddof=1) if len(v) > 1 else np.nan}
            for k, name in enumerate(["default_sens", "default_spec", "cleanthr_sens", "cleanthr_spec", "condthr_sens", "condthr_spec"]):
                r[name] = v[:, k].mean()
                r[name + "_sd"] = v[:, k].std(ddof=1) if len(v) > 1 else np.nan
            rows.append(r)
    pd.DataFrame(rows).to_csv(T("table_recalibration.csv"), index=False)


def roc_and_reliability():
    archs = [a for a in C.ARCHS if runs("main", a, "full", [C.SEEDS[0]])]
    if not archs:
        return
    sel = ["brightness", "darkening", "jpeg", "gaussian_noise"]
    sv = max(C.SEVERITIES)
    grid = np.linspace(0, 1, 201)
    fig, axes = plt.subplots(len(archs), len(sel), figsize=(3.4 * len(sel), 3.2 * len(archs)), squeeze=False)
    fig2, axes2 = plt.subplots(1, len(archs), figsize=(4.2 * len(archs), 4), squeeze=False)
    for i, a in enumerate(archs):
        rs = runs("main", a, "full")
        y = next(iter(rs.values()))["y/test"]
        pos = y >= C.REFERABLE

        def mroc(key):
            ts = []
            for r in rs.values():
                sc = M.softmax(r[key])[:, 2:].sum(1)
                o = np.argsort(-sc)
                tp = np.r_[0, np.cumsum(pos[o])] / pos.sum()
                fp = np.r_[0, np.cumsum(~pos[o])] / (~pos).sum()
                ts.append(np.interp(grid, fp, tp))
            return np.mean(ts, 0)
        base = mroc("test/clean/0")
        for j, c in enumerate(sel):
            ax = axes[i, j]
            ax.plot(grid, base, color="grey", ls="--", label="clean")
            ax.plot(grid, mroc(f"test/{c}/{sv}"), color=COLORS[c], label=f"{NICE[c]} (sev {sv})")
            ax.plot([0, 1], [0, 1], color="k", lw=.5)
            ax.set_xlabel("1 - specificity")
            if j == 0:
                ax.set_ylabel(f"{C.label(a)}\nsensitivity")
            ax.legend(frameon=False, fontsize=7, loc="lower right")
        ax = axes2[0, i]
        ax.plot([0, 1], [0, 1], "--", color="grey", label="perfect calibration")
        for c, col in [("clean", "k"), ("brightness", COLORS["brightness"]), ("jpeg", COLORS["jpeg"]), ("gaussian_noise", COLORS["gaussian_noise"])]:
            key = "test/clean/0" if c == "clean" else f"test/{c}/{sv}"
            P = np.concatenate([M.softmax(r[key]) for r in rs.values()])
            yy = np.concatenate([y for _ in rs])
            conf, acc = P.max(1), (P.argmax(1) == yy).astype(float)
            b = np.clip(np.ceil(conf * 10).astype(int) - 1, 0, 9)
            xs = [conf[b == k].mean() for k in range(10) if (b == k).sum() >= 5]
            ys = [acc[b == k].mean() for k in range(10) if (b == k).sum() >= 5]
            ece = np.mean([M.point_metrics(M.softmax(r[key]), y)["ece"] for r in rs.values()])
            ax.plot(xs, ys, marker="o", color=col, label=f"{NICE[c]} (ECE={ece:.3f})")
        ax.set_title(C.label(a))
        ax.set_xlabel("Predicted confidence")
        ax.set_ylabel("Empirical accuracy")
        ax.legend(frameon=False, fontsize=7)
    fig.tight_layout()
    save_fig(fig, "fig_roc_referable")
    fig2.tight_layout()
    save_fig(fig2, "fig_reliability")


def label_noise():
    rows, byg = [], []
    rng = np.random.default_rng(3)
    sv = max(C.SEVERITIES)
    for a in C.ARCHS:
        rs = runs("main", a, "full")
        if not rs:
            continue
        y = next(iter(rs.values()))["y/test"]
        conds = [("clean", 0)] + [(c, sv) for c in C.CORRUPTIONS + C.COMPOUNDS]
        P = {cs: probs(rs, f"test/{cs[0]}/{cs[1]}") for cs in conds}
        pred = {cs: p.argmax(-1) for cs, p in P.items()}
        for cs in conds:
            for g in range(C.NUM_CLASSES):
                mk = y == g
                if mk.any():
                    v = (pred[cs][:, mk] - g).mean(1)
                    byg.append({"arch": a, "corruption": cs[0], "severity": cs[1], "grade": g, "n": int(mk.sum()),
                                "msge_mean": v.mean(), "msge_sd": v.std(ddof=1) if len(v) > 1 else np.nan})
        for rate in [0.0] + C.LABEL_NOISE_RATES:
            reps = 1 if rate == 0 else C.LABEL_NOISE_REPS
            acc = {cs: [] for cs in conds}
            for _ in range(reps):
                yn = M.adjacent_noise(y, rate, rng) if rate else y
                for cs in conds:
                    e = np.mean([np.abs(M.ece_terms(P[cs][i], yn).sum(0)).sum() / len(yn) for i in range(len(P[cs]))])
                    acc[cs].append(((pred[cs] - yn).mean(), e))
            base = np.array(acc[("clean", 0)])
            for cs in conds[1:]:
                v = np.array(acc[cs])
                rows.append({"arch": a, "corruption": cs[0], "severity": cs[1], "noise_rate": rate,
                             "msge_clean": base[:, 0].mean(), "msge_corrupt": v[:, 0].mean(),
                             "delta_msge": (v[:, 0] - base[:, 0]).mean(), "delta_msge_sd": (v[:, 0] - base[:, 0]).std(),
                             "ece_clean": base[:, 1].mean(), "ece_corrupt": v[:, 1].mean(), "delta_ece": (v[:, 1] - base[:, 1]).mean()})
    pd.DataFrame(rows).to_csv(T("table_label_noise.csv"), index=False)
    pd.DataFrame(byg).to_csv(T("table_msge_by_grade.csv"), index=False)


def external():
    rows = []
    for name in ["idrid_grading", "messidor2"]:
        for a in C.ARCHS:
            rs = runs("main", a, "full")
            rs = {s: r for s, r in rs.items() if f"{name}/clean/0" in r}
            if not rs:
                continue
            y = next(iter(rs.values()))[f"y/{name}"]
            w = W(name, len(y))
            s0 = M.stats(probs(rs, f"{name}/clean/0"), y, w)
            keys = [("clean", 0)] + [(c, s) for c in C.CORRUPTIONS for s in C.EXTERNAL_SEVERITIES]
            for c, s in keys:
                st = M.stats(probs(rs, f"{name}/{c}/{s}"), y, w)
                for m in MAIN_METRICS:
                    r = {"dataset": name, "arch": a, "corruption": c, "severity": s, "metric": m, **S(m, st[m])}
                    if c != "clean":
                        dd = M.summarize_diff(st[m], s0[m])
                        r.update({"delta": dd["mean"], "delta_lo": dd["lo"], "delta_hi": dd["hi"], "delta_p": dd["p"]})
                    rows.append(r)
    if rows:
        pd.DataFrame(rows).to_csv(T("table_external.csv"), index=False)


def training_summary():
    rows, hist = [], []
    for p in glob.glob(os.path.join(C.RUNS, "*", "*", "*", "*", "*", "done.json")):
        with open(p) as f:
            d = json.load(f)
        rows.append(d)
        h = pd.read_csv(os.path.join(os.path.dirname(p), "history.csv"))
        for k, v in d.items():
            if k in ("protocol", "arch", "domain", "lr", "seed"):
                h[k] = v
        hist.append(h)
    if not rows:
        return
    df = pd.DataFrame(rows)
    df.to_csv(T("table_runs.csv"), index=False)
    g = df.groupby(["protocol", "arch", "domain", "lr"]).agg(n=("seed", "count"), best_epoch_mean=("best_epoch", "mean"),
                                                            best_epoch_sd=("best_epoch", "std"), epochs_run_mean=("epochs_run", "mean"),
                                                            best_val_qwk_mean=("best_val_qwk", "mean"), best_val_qwk_sd=("best_val_qwk", "std"),
                                                            train_minutes=("train_seconds", lambda x: x.mean() / 60)).reset_index()
    g.to_csv(T("table_training_summary.csv"), index=False)
    H = pd.concat(hist)
    doms = ["full", "high", "low"]
    fig, axes = plt.subplots(len(doms), len(C.ARCHS), figsize=(4 * len(C.ARCHS), 2.8 * len(doms)), squeeze=False)
    for i, d in enumerate(doms):
        for j, a in enumerate(C.ARCHS):
            ax = axes[i, j]
            for protocol, col in [("main", "#1f77b4"), ("v1", "#d62728")]:
                lr = lr_of(protocol, a) if (protocol != "main" or a in LR) else None
                h = H[(H.protocol == protocol) & (H.arch == a) & (H.domain == d)]
                if lr is not None:
                    h = h[np.isclose(h.lr, lr)]
                if not len(h):
                    continue
                q = h.groupby("epoch")["val_qwk"].agg(["mean", "std", "count"]).reset_index()
                ax.plot(q["epoch"], q["mean"], color=col, label=f"{protocol} protocol")
                ax.fill_between(q["epoch"], q["mean"] - q["std"].fillna(0), q["mean"] + q["std"].fillna(0), color=col, alpha=.15, lw=0)
            if i == 0:
                ax.set_title(C.label(a))
            if j == 0:
                ax.set_ylabel(f"{d} training\nvalidation QWK")
            if i == len(doms) - 1:
                ax.set_xlabel("Epoch")
            ax.grid(alpha=.25)
    axes[0, 0].legend(frameon=False, fontsize=8)
    fig.tight_layout()
    save_fig(fig, "fig_learning_curves")


def xai_tables():
    fs = glob.glob(os.path.join(C.RUNS, "xai", "*", "seed*", "xai.csv"))
    if not fs:
        return
    df = pd.concat([pd.read_csv(f).assign(arch=f.split(os.sep)[-3], seed=int(f.split(os.sep)[-2][4:])) for f in fs])
    ap = df[df.dataset == "aptos"].copy()
    ap["grade_group"] = ap["grade"].map({0: "0", 1: "1", 2: "2", 3: "3-4", 4: "3-4"})
    rows = []
    for gg in ["all", "0", "1", "2", "3-4"]:
        sub = ap if gg == "all" else ap[ap.grade_group == gg]
        for (a, c, s), g in sub.groupby(["arch", "corruption", "severity"]):
            per = g.groupby("seed")["energy_on_retina"].mean()
            img = g.groupby("image")["energy_on_retina"].mean()
            rows.append({"arch": a, "corruption": c, "severity": s, "grade_group": gg, "n_images": img.size,
                         "mean": per.mean(), "sd_seeds": per.std(ddof=1) if len(per) > 1 else np.nan,
                         "p05_images": img.quantile(.05), "frac_below_0.9": (img < .9).mean()})
    pd.DataFrame(rows).to_csv(T("table_xai_energy.csv"), index=False)
    idr = df[df.dataset == "idrid_seg"]
    if len(idr):
        rows = []
        for (a, c, s), g in idr.groupby(["arch", "corruption", "severity"]):
            r = {"arch": a, "corruption": c, "severity": s, "n_images": g["image"].nunique()}
            for m in ["lesion_lift", "lesion_fraction", "pointing_hit", "pixel_auc", "energy_on_retina", "lift_MA", "lift_HE", "lift_EX", "lift_SE"]:
                per = g.groupby("seed")[m].mean()
                r[m] = per.mean()
                r[m + "_sd"] = per.std(ddof=1) if len(per) > 1 else np.nan
            img = g.groupby("image")["lesion_lift"].mean().dropna()
            r["lift_gt1_wilcoxon_p"] = float(sst.wilcoxon(img - 1).pvalue) if len(img) > 5 else np.nan
            rows.append(r)
        pd.DataFrame(rows).to_csv(T("table_xai_lesion.csv"), index=False)


def xai_figs(meta, arr, splits):
    archs = [a for a in C.ARCHS if os.path.exists(os.path.join(C.RUNS, "xai", a, "seed0", "cams.npz"))]
    if not archs:
        return
    z = {a: np.load(os.path.join(C.RUNS, "xai", a, "seed0", "cams.npz")) for a in archs}
    ids = list(z[archs[0]]["aptos/ids"])
    idx = {k: i for i, k in enumerate(meta["id_code"])}
    imgs = [D.center_crop(np.asarray(arr[idx[k]])) for k in ids]
    grades = list(z[archs[0]]["aptos/grades"])
    fig, axes = plt.subplots(1 + len(archs), len(ids), figsize=(2.6 * len(ids), 2.7 * (1 + len(archs))), squeeze=False)
    for j, im in enumerate(imgs):
        axes[0, j].imshow(im)
        axes[0, j].set_title(f"grade {grades[j]}", fontsize=9)
        for i, a in enumerate(archs):
            axes[i + 1, j].imshow(im)
            axes[i + 1, j].imshow(z[a]["aptos/clean/0"][j].astype(np.float32), cmap="jet", alpha=.45)
            if j == 0:
                axes[i + 1, j].set_ylabel(C.label(a))
    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
    fig.tight_layout()
    save_fig(fig, "fig_xai_examples")
    sv = max(C.XAI_SEVERITIES)
    j = next((k for k, g in enumerate(grades) if g >= C.REFERABLE), 0)
    conds = [("clean", 0)] + [(c, sv) for c in C.CORRUPTIONS]
    fig, axes = plt.subplots(1 + len(archs), len(conds), figsize=(2.4 * len(conds), 2.5 * (1 + len(archs))), squeeze=False)
    raw = np.asarray(arr[idx[ids[j]]])
    for k, (c, s) in enumerate(conds):
        im = D.center_crop(raw if c == "clean" else corrupt(raw, c, s, ids[j]))
        axes[0, k].imshow(im)
        axes[0, k].set_title(NICE[c] + ("" if c == "clean" else f" ({s})"), fontsize=8)
        for i, a in enumerate(archs):
            axes[i + 1, k].imshow(im)
            axes[i + 1, k].imshow(z[a][f"aptos/{c}/{s}"][j].astype(np.float32), cmap="jet", alpha=.45)
            if k == 0:
                axes[i + 1, k].set_ylabel(C.label(a))
    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
    fig.tight_layout()
    save_fig(fig, "fig_xai_corrupted")
    sm, sa = D.load_external("idrid_seg")
    if sm is not None and "idrid/ids" in z[archs[0]].files:
        masks = np.load(os.path.join(C.CACHE, "idrid_seg_masks_256.npy"), mmap_mode="r")
        n = len(z[archs[0]]["idrid/ids"])
        fig, axes = plt.subplots(n, 1 + len(archs), figsize=(2.8 * (1 + len(archs)), 2.8 * n), squeeze=False)
        for r in range(n):
            im = D.center_crop(np.asarray(sa[r]))
            u = D.center_crop(np.asarray(masks[r]).transpose(1, 2, 0)).any(-1)
            axes[r, 0].imshow(im)
            axes[r, 0].contour(u, levels=[.5], colors="w", linewidths=.6)
            for i, a in enumerate(archs):
                axes[r, i + 1].imshow(im)
                axes[r, i + 1].imshow(z[a]["idrid/clean/0"][r].astype(np.float32), cmap="jet", alpha=.45)
                axes[r, i + 1].contour(u, levels=[.5], colors="w", linewidths=.6)
                if r == 0:
                    axes[r, i + 1].set_title(C.label(a))
        axes[0, 0].set_title("IDRiD lesions")
        for ax in axes.ravel():
            ax.set_xticks([])
            ax.set_yticks([])
        fig.tight_layout()
        save_fig(fig, "fig_xai_idrid")


def severity_examples(meta, arr, splits):
    te = splits["full"][2]
    i = next((t for t in te if meta["diagnosis"][t] >= C.REFERABLE), te[0])
    raw, key = np.asarray(arr[i]), meta["id_code"][i]
    cs = C.CORRUPTIONS + C.COMPOUNDS
    fig, axes = plt.subplots(len(cs), len(C.SEVERITIES), figsize=(1.9 * len(C.SEVERITIES), 1.9 * len(cs)), squeeze=False)
    for r, c in enumerate(cs):
        for k, s in enumerate(C.SEVERITIES):
            axes[r, k].imshow(D.center_crop(corrupt(raw, c, s, key)))
            axes[r, k].set_xticks([])
            axes[r, k].set_yticks([])
            if r == 0:
                axes[r, k].set_title(f"severity {s}", fontsize=8)
            if k == 0:
                axes[r, k].set_ylabel(NICE[c], fontsize=7)
    fig.tight_layout()
    save_fig(fig, "fig_severity_examples")


def latex():
    def tex(df, name, cols, header):
        lines = ["\\begin{tabular}{" + "l" * 2 + "r" * (len(cols) - 2) + "}", "\\hline", " & ".join(header) + " \\\\", "\\hline"]
        for _, r in df.iterrows():
            lines.append(" & ".join(str(r[c]) for c in cols) + " \\\\")
        lines += ["\\hline", "\\end{tabular}"]
        with open(T(name), "w") as f:
            f.write("\n".join(lines))
    p = T("table_benchmark.csv")
    if os.path.exists(p):
        b = pd.read_csv(p)
        b = b[b.protocol == "main"]
        rows = []
        for m in ["accuracy", "qwk", "macro_f1", "referable_sens", "referable_spec", "ref_auc", "auc_macro", "ece", "ece_ts", "msge"]:
            r = {"metric": m.replace("_", " ")}
            for a in C.ARCHS:
                x = b[(b.arch == a) & (b.metric == m)]
                r[a] = "" if not len(x) else (f"{x.iloc[0]['mean']:.3f} $\\pm$ {x.iloc[0]['sd']:.3f}" if np.isnan(x.iloc[0]["lo"]) else
                                             f"{x.iloc[0]['mean']:+.3f} ({x.iloc[0]['lo']:+.3f}, {x.iloc[0]['hi']:+.3f})" if m == "msge" else
                                             f"{x.iloc[0]['mean']:.3f} ({x.iloc[0]['lo']:.3f}--{x.iloc[0]['hi']:.3f})")
            rows.append(r)
        tex(pd.DataFrame(rows), "table_benchmark.tex", ["metric"] + C.ARCHS, ["Metric"] + [C.label(a) for a in C.ARCHS])
    p = T("table_generalisation_gap.csv")
    if os.path.exists(p):
        g = pd.read_csv(p)
        g = g[(g.split == "median")]
        g["arch"] = g["arch"].map(C.label)
        for c in ["in_domain_qwk", "cross_domain_qwk"]:
            g[c] = g[c].map(lambda v: f"{v:.3f}")
        g["gap_s"] = g.apply(lambda r: f"{r['gap']:+.3f} ({r['gap_lo']:+.3f}, {r['gap_hi']:+.3f})", axis=1)
        tex(g, "table_generalisation_gap.tex", ["protocol", "arch", "train", "in_domain_qwk", "cross_domain_qwk", "gap_s"],
            ["Protocol", "Model", "Train", "In-domain QWK", "Cross-domain QWK", "Gap (95\\% CI)"])
    p = T("table_robustness_sev5.csv")
    if os.path.exists(p):
        r = pd.read_csv(p)
        r = r[r.protocol == "main"].copy()
        r["arch"] = r["arch"].map(C.label)
        r["corruption"] = r["corruption"].map(NICE)
        for c in ["qwk", "referable_sens", "referable_spec", "ref_auc", "ece"]:
            r[c] = r[c].map(lambda v: f"{v:.3f}")
        r["msge"] = r["msge"].map(lambda v: f"{v:+.3f}")
        tex(r, "table_robustness_sev5.tex", ["arch", "corruption", "qwk", "referable_sens", "referable_spec", "ref_auc", "ece", "msge"],
            ["Model", "Condition", "QWK", "Ref. sens.", "Ref. spec.", "Ref. AUC", "ECE", "MSGE"])


def run_all():
    C.ensure_dirs()
    p = os.path.join(C.OUT, "lr_selection.json")
    if os.path.exists(p):
        with open(p) as f:
            LR.update(json.load(f))
    cache = {}

    def L():
        if not cache:
            cache["v"] = D.load_aptos()
        return cache["v"]
    steps = [("quality", lambda: quality_tables(L()[0])), ("severity mapping", lambda: severity_mapping(*L())),
             ("severity examples", lambda: severity_examples(*L())), ("training summary", training_summary),
             ("benchmark", benchmark), ("cross-condition", lambda: crosscond(L()[0], L()[2])), ("robustness", robustness),
             ("recalibration", recalibration), ("roc/reliability", roc_and_reliability), ("label noise", label_noise),
             ("external", external), ("xai tables", xai_tables), ("xai figures", lambda: xai_figs(*L())), ("latex", latex)]
    only = os.environ.get("DRFM_STEPS")
    if only:
        keep = set(x.strip() for x in only.split(","))
        steps = [x for x in steps if x[0] in keep]
    for name, fn in steps:
        print("analyze:", name, flush=True)
        try:
            fn()
        except Exception:
            import traceback
            traceback.print_exc()
    pd.DataFrame(list(NONFINITE.values()), columns=["protocol", "arch", "domain", "seed", "n_keys", "n_rows", "keys"]).to_csv(
        T("table_nonfinite_runs.csv"), index=False)
    if only:
        return
    meta, arr, splits = L()
    info = {"lr_selection": LR, "sharpness_threshold": splits["threshold"], "tertile_thresholds": splits["tertile_thresholds"],
            "split_sizes": {k: [len(x) for x in splits[k]] for k in ["full", "high", "low", "thigh", "tlow"]},
            "quality_counts": meta["quality"].value_counts().to_dict(), "quick": C.QUICK, "seeds": C.SEEDS, "n_boot": C.N_BOOT}
    with open(os.path.join(C.OUT, "run_info.json"), "w") as f:
        json.dump(info, f, indent=1)
    for t in sorted(glob.glob(os.path.join(C.TABLES, "*.csv"))):
        print("=" * 70)
        print(os.path.basename(t))
        print("=" * 70)
        d = pd.read_csv(t)
        print(d.head(60).to_string(index=False))


In [ ]:
%%writefile /kaggle/working/dr-failure-modes/drfm/config.py
import os
import glob

QUICK = os.environ.get("DRFM_QUICK", "0") == "1"
WORK = os.environ.get("DRFM_WORK", "/kaggle/working/quick" if QUICK else "/kaggle/working")
INPUT = os.environ.get("DRFM_INPUT", "/kaggle/input")
OUT = os.path.join(WORK, "outputs")
CACHE = os.environ.get("DRFM_CACHE", os.path.join(WORK, "cache"))
CKPT = os.path.join(WORK, "checkpoints")
RUNS = os.path.join(OUT, "runs")
TABLES = os.path.join(OUT, "tables")
FIGS = os.path.join(OUT, "figures")
CLAIMS = os.path.join(WORK, "claims")
LOGS = os.path.join(WORK, "logs")

SHORT = 256
IMG = 224
NUM_CLASSES = 5
REFERABLE = 2
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

ARCHS = os.environ.get("DRFM_ARCHS", "efficientnet_b0,resnet50,vit_small_patch16_224").split(",")
ARCH_LABEL = {"efficientnet_b0": "EfficientNet-B0", "resnet50": "ResNet-50", "vit_small_patch16_224": "ViT-Small"}


def label(a):
    return ARCH_LABEL.get(a, a)


SEEDS = [0, 1, 2, 3, 4]
TERTILE_SEEDS = [0, 1, 2]
SPLIT_SEED = 0
LR_GRID = [3e-5, 1e-4, 3e-4]

PROTOCOLS = {
    "main": {"epochs": 30, "warmup": 2, "patience": 8, "weight_decay": 1e-4, "batch": 64},
    "v1": {"epochs": 10, "warmup": 0, "patience": 0, "weight_decay": 1e-4, "batch": 64, "lr": 3e-4},
}

CORRUPTIONS = ["gaussian_blur", "brightness", "darkening", "contrast", "gaussian_noise", "jpeg"]
COMPOUNDS = ["compound_blur_dark_jpeg", "compound_noise_contrast_jpeg", "compound_field"]
NATIVE_CORRUPTIONS = ["gaussian_blur", "brightness", "contrast", "gaussian_noise", "jpeg"]
SEVERITIES = [1, 2, 3, 4, 5]
XAI_SEVERITIES = [3, 5]
EXTERNAL_SEVERITIES = [3, 5]

PARAMS = {
    "gaussian_blur": [0.5, 1.0, 1.5, 2.0, 3.0],
    "brightness": [1.2, 1.4, 1.6, 1.8, 2.1],
    "darkening": [0.8, 0.65, 0.5, 0.4, 0.3],
    "contrast": [0.85, 0.7, 0.55, 0.4, 0.3],
    "gaussian_noise": [3, 6, 10, 15, 22],
    "jpeg": [60, 45, 32, 22, 14],
}
NATIVE_PARAMS = {
    "gaussian_blur": [0.5, 1.0, 2.0, 3.0, 4.0],
    "brightness": [1.2, 1.4, 1.6, 1.8, 2.1],
    "contrast": [0.85, 0.7, 0.55, 0.4, 0.3],
    "gaussian_noise": [8, 16, 28, 42, 60],
    "jpeg": [60, 45, 32, 22, 14],
}

N_BOOT = 2000
ALPHA = 0.05
AUC_MARGIN = 0.02
SHIFT_MARGIN = 0.05
LABEL_NOISE_RATES = [0.1, 0.2, 0.3]
LABEL_NOISE_REPS = 200
GRADE_MATCH_REPS = 200
PREVALENCE_REF = 406
PREVALENCE_NONREF = 594
WORKERS = int(os.environ.get("DRFM_WORKERS", "2"))
EVAL_BATCH = 128

if QUICK:
    SEEDS = [0]
    TERTILE_SEEDS = [0]
    LR_GRID = [1e-4]
    SEVERITIES = [1, 5]
    XAI_SEVERITIES = [5]
    EXTERNAL_SEVERITIES = [5]
    PROTOCOLS["main"].update({"epochs": 2, "warmup": 1, "patience": 0})
    PROTOCOLS["v1"].update({"epochs": 2})
    N_BOOT = 200
    LABEL_NOISE_REPS = 20
    GRADE_MATCH_REPS = 20

QUICK_N = int(os.environ.get("DRFM_QUICK_N", "400"))


def find_aptos():
    env = os.environ.get("DRFM_APTOS")
    if env:
        return env
    for c in sorted(glob.glob(os.path.join(INPUT, "**", "train.csv"), recursive=True)):
        d = os.path.dirname(c)
        if os.path.isdir(os.path.join(d, "train_images")):
            return d
    raise FileNotFoundError("APTOS 2019 not found under " + INPUT)


def ensure_dirs():
    for d in [OUT, CACHE, CKPT, RUNS, TABLES, FIGS, CLAIMS, LOGS]:
        os.makedirs(d, exist_ok=True)


In [ ]:
%%writefile /kaggle/working/dr-failure-modes/drfm/corruptions.py
import io
import zlib

import numpy as np
from PIL import Image, ImageEnhance, ImageFilter

from . import config as C


def rng_for(key, ctype, s):
    return np.random.default_rng(zlib.crc32(f"{key}|{ctype}|{s}".encode()))


def _apply(pil, ctype, v, rng):
    if ctype == "gaussian_blur":
        return pil.filter(ImageFilter.GaussianBlur(v))
    if ctype in ("brightness", "darkening"):
        return ImageEnhance.Brightness(pil).enhance(v)
    if ctype == "contrast":
        return ImageEnhance.Contrast(pil).enhance(v)
    if ctype == "gaussian_noise":
        a = np.asarray(pil).astype(np.float32) + rng.normal(0, v, (pil.size[1], pil.size[0], 3)).astype(np.float32)
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    if ctype == "jpeg":
        b = io.BytesIO()
        pil.save(b, "JPEG", quality=int(v))
        b.seek(0)
        return Image.open(b).convert("RGB")
    raise ValueError(ctype)


def _p(ctype, s):
    return C.PARAMS[ctype][s - 1]


def _compound(pil, ctype, s, rng):
    if ctype == "compound_blur_dark_jpeg":
        for t in ["darkening", "gaussian_blur", "jpeg"]:
            pil = _apply(pil, t, _p(t, s), rng)
        return pil
    if ctype == "compound_noise_contrast_jpeg":
        for t in ["contrast", "gaussian_noise", "jpeg"]:
            pil = _apply(pil, t, _p(t, s), rng)
        return pil
    if ctype == "compound_field":
        lv = lambda: int(rng.integers(1, s + 1))
        if rng.random() < 0.7:
            t = "brightness" if rng.random() < 0.5 else "darkening"
            pil = _apply(pil, t, _p(t, lv()), rng)
        if rng.random() < 0.5:
            pil = _apply(pil, "contrast", _p("contrast", lv()), rng)
        if rng.random() < 0.7:
            pil = _apply(pil, "gaussian_blur", _p("gaussian_blur", lv()), rng)
        if rng.random() < 0.5:
            pil = _apply(pil, "gaussian_noise", _p("gaussian_noise", lv()), rng)
        return _apply(pil, "jpeg", _p("jpeg", lv()), rng)
    raise ValueError(ctype)


def corrupt(img, ctype, s, key):
    if ctype in (None, "clean") or s == 0:
        return img
    rng = rng_for(key, ctype, s)
    pil = Image.fromarray(np.asarray(img))
    out = _compound(pil, ctype, s, rng) if ctype.startswith("compound") else _apply(pil, ctype, _p(ctype, s), rng)
    return np.asarray(out, dtype=np.uint8)


def corrupt_native(img, ctype, s, key):
    rng = rng_for(key, "native_" + ctype, s)
    return np.asarray(_apply(Image.fromarray(img), ctype, C.NATIVE_PARAMS[ctype][s - 1], rng), dtype=np.uint8)


def conditions(include_compound=True):
    out = [("clean", 0)]
    for c in C.CORRUPTIONS:
        for s in C.SEVERITIES:
            out.append((c, s))
    if include_compound:
        for c in C.COMPOUNDS:
            for s in C.SEVERITIES:
                out.append((c, s))
    return out


In [ ]:
%%writefile /kaggle/working/dr-failure-modes/drfm/data.py
import os
import glob
import json
from multiprocessing import Pool

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

from . import config as C
from .corruptions import corrupt_native

IMG_EXT = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")


def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        return np.array(Image.open(path).convert("RGB"))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def read_gray(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        img = np.array(Image.open(path).convert("L"))
    return img


def to_square(img, size=C.SHORT, interp=cv2.INTER_AREA):
    h, w = img.shape[:2]
    s = size / min(h, w)
    nw, nh = max(size, int(round(w * s))), max(size, int(round(h * s)))
    r = cv2.resize(img, (nw, nh), interpolation=interp)
    y0, x0 = (nh - size) // 2, (nw - size) // 2
    return np.ascontiguousarray(r[y0:y0 + size, x0:x0 + size])


def center_crop(arr, size=C.IMG):
    h, w = arr.shape[-3], arr.shape[-2]
    y0, x0 = (h - size) // 2, (w - size) // 2
    return arr[..., y0:y0 + size, x0:x0 + size, :]


def quality_metrics(gray):
    g = gray.astype(np.float64)
    lap = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    gx = cv2.Sobel(g, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(g, cv2.CV_64F, 0, 1, ksize=3)
    ten = float(np.mean(gx ** 2 + gy ** 2))
    bren = float(np.mean((g[:, 2:] - g[:, :-2]) ** 2))
    mask = g > 10
    fov = float(mask.mean())
    vals = g[mask] if mask.any() else g.ravel()
    return {"lap_var": lap, "tenengrad": ten, "brenner": bren, "fov": fov,
            "brightness": float(vals.mean() / 255.0), "rms_contrast": float(vals.std() / 255.0)}


def v1_gray(path):
    return cv2.resize(read_gray(path), (256, 256))


def load_aptos_frame():
    root = C.find_aptos()
    df = pd.read_csv(os.path.join(root, "train.csv"))
    df["path"] = df["id_code"].apply(lambda c: os.path.join(root, "train_images", c + ".png"))
    df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)
    if C.QUICK and len(df) > C.QUICK_N:
        df, _ = train_test_split(df, train_size=C.QUICK_N, stratify=df["diagnosis"], random_state=0)
        df = df.sort_index().reset_index(drop=True)
    return df


def _strat(y):
    return y if y.value_counts().min() >= 2 else None


def stratified_split(frame, seed=C.SPLIT_SEED):
    tr, tmp = train_test_split(frame, test_size=0.30, stratify=_strat(frame["diagnosis"]), random_state=seed)
    va, te = train_test_split(tmp, test_size=0.50, stratify=_strat(tmp["diagnosis"]), random_state=seed)
    return list(tr.index), list(va.index), list(te.index)


def _aptos_worker(args):
    path, native = args
    q = quality_metrics(v1_gray(path))
    rgb = read_rgb(path)
    q["native_h"], q["native_w"] = int(rgb.shape[0]), int(rgb.shape[1])
    sq = to_square(rgb)
    nat = None
    if native:
        nat = {}
        for c in C.NATIVE_CORRUPTIONS:
            for s in C.SEVERITIES:
                nat[(c, s)] = to_square(corrupt_native(rgb, c, s, os.path.basename(path)))
    return q, sq, nat


def build_aptos_cache():
    C.ensure_dirs()
    meta_path = os.path.join(C.CACHE, "aptos_meta.csv")
    if os.path.exists(meta_path) and os.path.exists(os.path.join(C.CACHE, "splits.json")):
        print("APTOS cache exists")
        return
    df = load_aptos_frame()
    full_tr, full_va, full_te = stratified_split(df)
    test_set = set(full_te)
    n = len(df)
    arr = np.lib.format.open_memmap(os.path.join(C.CACHE, "aptos_256.npy"), mode="w+", dtype=np.uint8, shape=(n, C.SHORT, C.SHORT, 3))
    nat_arrs = {}
    for c in C.NATIVE_CORRUPTIONS:
        for s in C.SEVERITIES:
            nat_arrs[(c, s)] = np.lib.format.open_memmap(os.path.join(C.CACHE, f"native_{c}_{s}.npy"), mode="w+", dtype=np.uint8, shape=(len(full_te), C.SHORT, C.SHORT, 3))
    te_pos = {i: k for k, i in enumerate(full_te)}
    rows = []
    jobs = [(p, i in test_set) for i, p in enumerate(df["path"])]
    with Pool(max(1, os.cpu_count() or 1)) as pool:
        for i, (q, sq, nat) in enumerate(pool.imap(_aptos_worker, jobs, chunksize=4)):
            arr[i] = sq
            if nat is not None:
                for key, v in nat.items():
                    nat_arrs[key][te_pos[i]] = v
            rows.append(q)
            if (i + 1) % 250 == 0:
                print(f"  cached {i + 1}/{n}", flush=True)
    arr.flush()
    for v in nat_arrs.values():
        v.flush()
    meta = pd.concat([df[["id_code", "diagnosis"]].reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    thr = float(meta["lap_var"].median())
    meta["quality"] = np.where(meta["lap_var"] >= thr, "high", "low")
    q33, q67 = meta["lap_var"].quantile([1 / 3, 2 / 3]).tolist()
    meta["tertile"] = np.where(meta["lap_var"] <= q33, "low", np.where(meta["lap_var"] >= q67, "high", "mid"))
    meta.to_csv(meta_path, index=False)
    ints = lambda parts: [[int(i) for i in p] for p in parts]
    splits = {"full": ints([full_tr, full_va, full_te]), "threshold": thr, "tertile_thresholds": [float(q33), float(q67)]}
    for q in ["high", "low"]:
        splits[q] = ints(stratified_split(meta[meta["quality"] == q]))
        splits["t" + q] = ints(stratified_split(meta[meta["tertile"] == q]))
    with open(os.path.join(C.CACHE, "splits.json"), "w") as f:
        json.dump(splits, f)
    print("Sharpness median threshold =", round(thr, 1))
    print(meta["quality"].value_counts().to_string())
    for k in ["full", "high", "low", "thigh", "tlow"]:
        print(k, [len(p) for p in splits[k]])


def load_aptos():
    meta = pd.read_csv(os.path.join(C.CACHE, "aptos_meta.csv"))
    arr = np.load(os.path.join(C.CACHE, "aptos_256.npy"), mmap_mode="r")
    with open(os.path.join(C.CACHE, "splits.json")) as f:
        splits = json.load(f)
    return meta, arr, splits


def load_native(c, s):
    return np.load(os.path.join(C.CACHE, f"native_{c}_{s}.npy"), mmap_mode="r")


def _images_under(root):
    out = {}
    for p in glob.glob(os.path.join(root, "**", "*"), recursive=True):
        if p.lower().endswith(IMG_EXT) and os.path.isfile(p):
            out.setdefault(os.path.splitext(os.path.basename(p))[0].lower(), p)
    return out


def _split_of(p):
    s = p.lower()
    return "test" if "test" in s else ("train" if "train" in s else "")


def find_idrid_grading():
    csvs = [p for p in glob.glob(os.path.join(C.INPUT, "**", "*.csv"), recursive=True) if "grading" in p.lower() and "label" in p.lower()]
    rows = []
    for p in csvs:
        try:
            d = pd.read_csv(p)
        except Exception:
            continue
        d.columns = [c.strip() for c in d.columns]
        if "Image name" in d.columns and "Retinopathy grade" in d.columns:
            d = d[["Image name", "Retinopathy grade"]].dropna()
            d.columns = ["image", "grade"]
            d["image"] = d["image"].astype(str).str.strip()
            d["split"] = _split_of(os.path.basename(p))
            d["root"] = os.path.dirname(os.path.dirname(p))
            rows.append(d)
    if not rows:
        return None
    d = pd.concat(rows).drop_duplicates(["split", "image"])
    imgs = {}
    for r in d["root"].unique():
        for p in glob.glob(os.path.join(r, "**", "*"), recursive=True):
            if p.lower().endswith(IMG_EXT) and os.path.isfile(p):
                rel = os.path.relpath(p, r)
                imgs.setdefault((_split_of(rel), os.path.splitext(os.path.basename(p))[0].lower()), p)
    anyname = {}
    for (sp, n), p in imgs.items():
        anyname.setdefault(n, p)
    d["path"] = [imgs.get((s, i.lower())) or (anyname.get(i.lower()) if not s else None) for s, i in zip(d["split"], d["image"])]
    d = d.dropna(subset=["path"])
    d["grade"] = d["grade"].astype(int)
    d["image"] = d["split"] + "_" + d["image"]
    return d[["image", "grade", "path"]].reset_index(drop=True) if len(d) else None


def find_messidor():
    aptos = C.find_aptos()
    for p in glob.glob(os.path.join(C.INPUT, "**", "*.csv"), recursive=True):
        if os.path.abspath(p).startswith(os.path.abspath(aptos)):
            continue
        try:
            d = pd.read_csv(p)
        except Exception:
            continue
        cols = {c.lower().strip(): c for c in d.columns}
        if not any("adjudicated" in c for c in cols):
            continue
        idc = next((cols[c] for c in ["image_id", "id_code", "image", "image name"] if c in cols), None)
        gc = next((cols[c] for c in ["adjudicated_dr_grade", "diagnosis", "dr_grade"] if c in cols), None)
        if idc is None or gc is None:
            continue
        if "adjudicated_gradable" in cols:
            d = d[d[cols["adjudicated_gradable"]] == 1]
        d = d[[idc, gc]].dropna()
        d.columns = ["image", "grade"]
        root = os.path.dirname(p)
        while root != os.path.dirname(root) and os.path.abspath(os.path.dirname(root)) != os.path.abspath(C.INPUT):
            root = os.path.dirname(root)
        imgs = _images_under(root)
        d["path"] = d["image"].astype(str).map(lambda s: imgs.get(os.path.splitext(os.path.basename(s))[0].lower()))
        d = d.dropna(subset=["path"])
        if len(d):
            d["grade"] = d["grade"].astype(int)
            return d.reset_index(drop=True)
    return None


def find_idrid_seg():
    masks = {}
    for t in ["MA", "HE", "EX", "SE"]:
        for p in glob.glob(os.path.join(C.INPUT, "**", f"*_{t}.tif*"), recursive=True):
            stem = os.path.basename(p).rsplit("_", 1)[0]
            masks.setdefault(stem, {})[t] = p
    if not masks:
        return None
    imgs = {}
    roots = {os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(p)))) for v in masks.values() for p in v.values()}
    for r in roots:
        imgs.update(_images_under(r))
    rows = []
    for stem, m in sorted(masks.items()):
        p = imgs.get(stem.lower())
        if p is not None and not p.lower().endswith((".tif", ".tiff")):
            rows.append({"image": stem, "path": p, **{t: m.get(t) for t in ["MA", "HE", "EX", "SE"]}})
    return pd.DataFrame(rows) if rows else None


def _ext_worker(path):
    return to_square(read_rgb(path))


def _mask_worker(row):
    out = []
    for t in ["MA", "HE", "EX", "SE"]:
        p = row.get(t)
        if isinstance(p, str):
            m = np.array(Image.open(p).convert("L")) > 0
            out.append(to_square(m.astype(np.float32), interp=cv2.INTER_AREA) > 0)
        else:
            out.append(None)
    out = [o if o is not None else np.zeros((C.SHORT, C.SHORT), bool) for o in out]
    return np.stack(out).astype(np.uint8)


def build_external_cache():
    C.ensure_dirs()
    found = {"idrid_grading": find_idrid_grading(), "messidor2": find_messidor()}
    for name, d in found.items():
        if d is None:
            print(f"{name}: not attached, skipped")
            continue
        if C.QUICK:
            d = d.head(60)
        npy = os.path.join(C.CACHE, f"{name}_256.npy")
        if not os.path.exists(npy):
            with Pool(max(1, os.cpu_count() or 1)) as pool:
                arr = np.stack(pool.map(_ext_worker, list(d["path"]), chunksize=4))
            np.save(npy, arr)
            d.to_csv(os.path.join(C.CACHE, f"{name}_meta.csv"), index=False)
        print(f"{name}: {len(d)} images, grades {d['grade'].value_counts().sort_index().to_dict()}")
    seg = find_idrid_seg()
    if seg is None:
        print("idrid_seg: not attached, skipped")
        return
    npy = os.path.join(C.CACHE, "idrid_seg_256.npy")
    if not os.path.exists(npy):
        with Pool(max(1, os.cpu_count() or 1)) as pool:
            arr = np.stack(pool.map(_ext_worker, list(seg["path"]), chunksize=2))
            masks = np.stack(pool.map(_mask_worker, seg.to_dict("records"), chunksize=2))
        np.save(npy, arr)
        np.save(os.path.join(C.CACHE, "idrid_seg_masks_256.npy"), masks)
        seg.to_csv(os.path.join(C.CACHE, "idrid_seg_meta.csv"), index=False)
    print(f"idrid_seg: {len(seg)} images with lesion masks")


def load_external(name):
    npy = os.path.join(C.CACHE, f"{name}_256.npy")
    if not os.path.exists(npy):
        return None, None
    return pd.read_csv(os.path.join(C.CACHE, f"{name}_meta.csv")), np.load(npy, mmap_mode="r")


In [ ]:
%%writefile /kaggle/working/dr-failure-modes/drfm/engine.py
import os
import json
import math
import time
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

from . import config as C
from . import data as D
from .corruptions import corrupt, conditions
from .metrics import softmax, from_conf

DEV = "cuda" if torch.cuda.is_available() else "cpu"
PRETRAINED = os.environ.get("DRFM_PRETRAINED", "1") == "1"


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


class ArrDS(Dataset):
    def __init__(self, arr, idx, labels, keys, train=False, ctype="clean", sev=0):
        self.arr, self.idx, self.labels, self.keys = arr, list(idx), list(labels), list(keys)
        self.train, self.ctype, self.sev = train, ctype, sev
        self.aug = v2.Compose([v2.RandomResizedCrop(C.IMG, scale=(0.8, 1.0), antialias=True), v2.RandomHorizontalFlip()])

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, k):
        img = np.asarray(self.arr[self.idx[k]])
        if self.ctype != "clean":
            img = corrupt(img, self.ctype, self.sev, self.keys[k])
        if self.train:
            x = self.aug(torch.from_numpy(np.array(img)).permute(2, 0, 1))
        else:
            x = torch.from_numpy(np.array(D.center_crop(img))).permute(2, 0, 1)
        return x, int(self.labels[k])


def _winit(wid):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


def make_loader(ds, batch, shuffle, seed=0):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, num_workers=C.WORKERS, pin_memory=DEV == "cuda",
                      drop_last=False, worker_init_fn=_winit, generator=g, persistent_workers=False)


_MEAN = torch.tensor(C.MEAN).view(1, 3, 1, 1) * 255
_STD = torch.tensor(C.STD).view(1, 3, 1, 1) * 255


def prep(x):
    x = x.to(DEV, non_blocking=True).float()
    return ((x - _MEAN.to(DEV)) / _STD.to(DEV)).contiguous(memory_format=torch.channels_last)


def build_model(arch):
    import timm
    m = timm.create_model(arch, pretrained=PRETRAINED, num_classes=C.NUM_CLASSES)
    return m.to(DEV).to(memory_format=torch.channels_last)


@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    out, ys = [], []
    for x, y in loader:
        xx = prep(x)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda"):
            z = model(xx).float()
        bad = ~torch.isfinite(z).all(1)
        if bad.any():
            z[bad] = model(xx[bad]).float()
        out.append(z.cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(out), np.concatenate(ys)


def _qwk(y, p):
    o = np.zeros((C.NUM_CLASSES, C.NUM_CLASSES))
    np.add.at(o, (y, p), 1)
    return float(from_conf(o)["qwk"])


def run_dir(job):
    return os.path.join(C.RUNS, job["protocol"], job["arch"], job["domain"], f"lr{job['lr']:g}", f"seed{job['seed']}")


def ckpt_path(arch, seed, lr):
    return os.path.join(C.CKPT, f"{arch}_lr{lr:g}_seed{seed}.pt")


def _sets(meta, splits, domain):
    tr, va, te = splits[domain]
    return tr, va, te


def train(job, meta, arr, splits):
    p = C.PROTOCOLS[job["protocol"]]
    set_seed(job["seed"])
    tr, va, _ = _sets(meta, splits, job["domain"])
    y = meta["diagnosis"].values
    keys = meta["id_code"].values
    model = build_model(job["arch"])
    counts = np.bincount(y[tr], minlength=C.NUM_CLASSES).astype(np.float64)
    w = counts.sum() / (counts + 1e-6)
    w = torch.tensor(w / w.mean(), dtype=torch.float32, device=DEV)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=job["lr"], weight_decay=p["weight_decay"])
    tl = make_loader(ArrDS(arr, tr, y[tr], keys[tr], train=True), p["batch"], True, job["seed"])
    vl = make_loader(ArrDS(arr, va, y[va], keys[va]), C.EVAL_BATCH, False)
    spe = len(tl)
    total = p["epochs"] * spe
    wu = p["warmup"] * spe

    def factor(step):
        if p["warmup"] == 0:
            return 0.5 * (1 + math.cos(math.pi * (step // spe) / p["epochs"]))
        if step < wu:
            return (step + 1) / wu
        return 0.5 * (1 + math.cos(math.pi * (step - wu) / max(1, total - wu)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, factor)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    best, best_ep, best_state, hist, bad = -2.0, -1, None, [], 0
    for ep in range(p["epochs"]):
        model.train()
        t0, tot, nb = time.time(), 0.0, 0
        for x, t in tl:
            t = t.to(DEV, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda"):
                loss = crit(model(prep(x)), t)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += float(loss.detach())
            nb += 1
        z, yv = predict_logits(model, vl)
        nonfinite = int((~np.isfinite(z).all(1)).sum())
        q = _qwk(yv, z.argmax(1)) if nonfinite == 0 else -1.0
        vloss = float(-np.mean(np.log(softmax(z)[np.arange(len(yv)), yv] + 1e-12))) if nonfinite == 0 else float("nan")
        hist.append({"epoch": ep + 1, "val_nonfinite": nonfinite, "train_loss": tot / max(1, nb), "val_loss": vloss, "val_qwk": q,
                     "lr": opt.param_groups[0]["lr"], "seconds": time.time() - t0})
        print(f"  [{job['id']}] epoch {ep + 1}/{p['epochs']} val_QWK={q:.4f} ({time.time() - t0:.0f}s)", flush=True)
        if q > best:
            best, best_ep, bad = q, ep + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if p["patience"] and bad >= p["patience"]:
                break
    model.load_state_dict(best_state)
    return model, hist, best, best_ep


def _eval_set(model, arr, idx, labels, keys, ctype="clean", sev=0):
    z, y = predict_logits(model, make_loader(ArrDS(arr, idx, labels, keys, ctype=ctype, sev=sev), C.EVAL_BATCH, False))
    return z


def evaluate(model, job, meta, arr, splits):
    y = meta["diagnosis"].values
    keys = meta["id_code"].values
    res = {}
    dom = job["domain"]
    if dom == "full":
        _, va, te = splits["full"]
        res["y/test"] = y[te]
        res["y/val"] = y[va]
        for c, s in conditions(True):
            res[f"test/{c}/{s}"] = _eval_set(model, arr, te, y[te], keys[te], c, s)
            if job["protocol"] == "main" or c == "clean":
                res[f"val/{c}/{s}"] = _eval_set(model, arr, va, y[va], keys[va], c, s)
        pos = np.arange(len(te))
        for c in C.NATIVE_CORRUPTIONS:
            for s in C.SEVERITIES:
                res[f"native/{c}/{s}"] = _eval_set(model, D.load_native(c, s), pos, y[te], keys[te])
        if job["protocol"] == "main":
            for name in ["idrid_grading", "messidor2"]:
                em, ea = D.load_external(name)
                if em is None:
                    continue
                ey, ek, ei = em["grade"].values, em["image"].astype(str).values, np.arange(len(em))
                res[f"y/{name}"] = ey
                res[f"{name}/clean/0"] = _eval_set(model, ea, ei, ey, ek)
                for c in C.CORRUPTIONS:
                    for s in C.EXTERNAL_SEVERITIES:
                        res[f"{name}/{c}/{s}"] = _eval_set(model, ea, ei, ey, ek, c, s)
    else:
        pair = ["high", "low"] if dom in ("high", "low") else ["thigh", "tlow"]
        _, va, _ = splits[dom]
        res["y/val"] = y[va]
        res["val/clean/0"] = _eval_set(model, arr, va, y[va], keys[va])
        for d in pair:
            te = splits[d][2]
            res[f"y/{d}_test"] = y[te]
            res[f"{d}_test/clean/0"] = _eval_set(model, arr, te, y[te], keys[te])
    return res


def run_train_job(job):
    out = run_dir(job)
    if os.path.exists(os.path.join(out, "done.json")):
        return
    os.makedirs(out, exist_ok=True)
    meta, arr, splits = D.load_aptos()
    t0 = time.time()
    model, hist, best, best_ep = train(job, meta, arr, splits)
    t1 = time.time()
    if job["protocol"] == "main" and job["domain"] == "full":
        os.makedirs(C.CKPT, exist_ok=True)
        torch.save({k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()},
                   ckpt_path(job["arch"], job["seed"], job["lr"]))
    res = evaluate(model, job, meta, arr, splits)
    np.savez_compressed(os.path.join(out, "logits.npz"), **res)
    import pandas as pd
    pd.DataFrame(hist).to_csv(os.path.join(out, "history.csv"), index=False)
    with open(os.path.join(out, "done.json"), "w") as f:
        json.dump({**job, "best_val_qwk": best, "best_epoch": best_ep, "epochs_run": len(hist),
                   "train_seconds": t1 - t0, "eval_seconds": time.time() - t1}, f)
    del model
    if DEV == "cuda":
        torch.cuda.empty_cache()


In [ ]:
%%writefile /kaggle/working/dr-failure-modes/drfm/metrics.py
import numpy as np
from scipy import stats as st
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score

from . import config as C

K = C.NUM_CLASSES
_I, _J = np.meshgrid(np.arange(K), np.arange(K), indexing="ij")
_W2 = (_I - _J) ** 2 / (K - 1) ** 2


def softmax(z, t=1.0):
    z = np.asarray(z, dtype=np.float64) / t
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)


def boot_weights(n, reps=None, seed=0):
    reps = C.N_BOOT if reps is None else reps
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, (reps, n))
    w = np.stack([np.bincount(r, minlength=n) for r in idx]).astype(np.float32)
    return np.vstack([np.ones((1, n), np.float32), w])


def _div(a, b):
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(b > 0, a / np.where(b > 0, b, 1), np.nan)


def from_conf(o):
    o = o.astype(np.float64)
    n = o.sum((-1, -2))
    row, col = o.sum(-1), o.sum(-2)
    diag = np.diagonal(o, axis1=-2, axis2=-1)
    e = row[..., :, None] * col[..., None, :] / np.where(n > 0, n, 1)[..., None, None]
    num, den = (_W2 * o).sum((-1, -2)), (_W2 * e).sum((-1, -2))
    rec, prec = _div(diag, row), _div(diag, col)
    rec0, prec0 = np.nan_to_num(rec), np.nan_to_num(prec)
    f1 = np.nan_to_num(_div(2 * rec0 * prec0, rec0 + prec0))
    out = {
        "accuracy": _div(diag.sum(-1), n),
        "qwk": 1 - _div(num, den),
        "macro_f1": f1.mean(-1),
        "referable_sens": _div(o[..., C.REFERABLE:, C.REFERABLE:].sum((-1, -2)), o[..., C.REFERABLE:, :].sum((-1, -2))),
        "referable_spec": _div(o[..., :C.REFERABLE, :C.REFERABLE].sum((-1, -2)), o[..., :C.REFERABLE, :].sum((-1, -2))),
        "msge": _div(((_J - _I) * o).sum((-1, -2)), n),
    }
    for k in range(K):
        out[f"recall_{k}"] = rec[..., k]
    return out


def weighted_auc(score, pos, w):
    order = np.argsort(score, kind="mergesort")
    s, p = score[order], pos[order].astype(np.float32)
    ww = w[:, order]
    starts = np.flatnonzero(np.r_[True, s[1:] != s[:-1]])
    wp = np.add.reduceat(ww * p, starts, axis=1)
    wn = np.add.reduceat(ww * (1 - p), starts, axis=1)
    below = np.cumsum(wn, axis=1) - wn
    num = (wp * (below + 0.5 * wn)).sum(1)
    return _div(num, wp.sum(1) * wn.sum(1))


def ece_terms(probs, y, bins=10):
    conf = probs.max(-1)
    correct = (probs.argmax(-1) == y).astype(np.float64)
    b = np.clip(np.ceil(conf * bins).astype(int) - 1, 0, bins - 1)
    g = np.zeros(conf.shape + (bins,))
    np.put_along_axis(g, b[..., None], (correct - conf)[..., None], axis=-1)
    return g


def stats(probs, y, w):
    probs = np.asarray(probs, dtype=np.float64)
    if probs.ndim == 2:
        probs = probs[None]
    y = np.asarray(y).astype(int)
    pred = probs.argmax(-1)
    cell = y[None] * K + pred
    m = np.zeros(cell.shape + (K * K,), np.float32)
    np.put_along_axis(m, cell[..., None], 1.0, axis=-1)
    o = np.einsum("rn,snk->srk", w, m).reshape(probs.shape[0], w.shape[0], K, K)
    out = from_conf(o)
    n = w.sum(1)[None]
    out["ece"] = np.abs(np.einsum("rn,snb->srb", w, ece_terms(probs, y).astype(np.float32))).sum(-1) / n
    pos = y >= C.REFERABLE
    ref = probs[..., C.REFERABLE:].sum(-1)
    out["ref_auc"] = np.stack([weighted_auc(ref[s], pos, w) for s in range(probs.shape[0])])
    out["mean_pred"] = (w @ pred.T.astype(np.float32)).T / n
    return out


def shift_stats(pred, pred_clean, w):
    d = (pred - pred_clean).astype(np.float32)
    n = w.sum(1)[None]
    return {"pred_shift": (w @ d.T).T / n, "up_rate": (w @ (d > 0).T.astype(np.float32)).T / n,
            "down_rate": (w @ (d < 0).T.astype(np.float32)).T / n}


def point_metrics(probs, y):
    out = {k: float(v[0, 0]) for k, v in stats(probs, y, np.ones((1, len(y)), np.float32)).items()}
    try:
        present = [c for c in range(K) if (y == c).sum() > 0]
        yoh = np.eye(K)[y][:, present]
        out["auc_macro"] = float(roc_auc_score(yoh, probs[:, present], average="macro", multi_class="ovr"))
    except Exception:
        out["auc_macro"] = float("nan")
    return out


def summarize(arr, basic=False):
    arr = np.asarray(arr, dtype=np.float64)
    m = np.nanmean(arr, 0)
    pt, rep = m[0], m[1:]
    lo, hi = np.nanpercentile(rep, [2.5, 97.5]) if np.isfinite(rep).any() else (np.nan, np.nan)
    if basic:
        bias = float(np.nanmean(rep)) - pt
        lo, hi = max(0.0, lo - bias), hi - bias
    sd = float(np.nanstd(arr[:, 0], ddof=1)) if arr.shape[0] > 1 else float("nan")
    return {"mean": float(pt), "sd": sd, "lo": float(lo), "hi": float(hi)}


def boot_p(d):
    d = np.asarray(d, dtype=np.float64)[1:]
    d = d[np.isfinite(d)]
    if not len(d):
        return float("nan")
    return float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))


def summarize_diff(a, b):
    d = np.nanmean(np.asarray(a, np.float64), 0) - np.nanmean(np.asarray(b, np.float64), 0)
    lo, hi = np.nanpercentile(d[1:], [2.5, 97.5])
    return {"mean": float(d[0]), "lo": float(lo), "hi": float(hi), "p": boot_p(d)}


def welch(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 2 or len(b) < 2:
        return float("nan")
    return float(st.ttest_ind(a, b, equal_var=False).pvalue)


def holm(p):
    p = np.asarray(p, dtype=np.float64)
    out = np.full_like(p, np.nan)
    ok = np.flatnonzero(np.isfinite(p))
    if not len(ok):
        return out
    order = ok[np.argsort(p[ok])]
    m = len(order)
    run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (m - r) * p[i]))
        out[i] = run
    return out


def fit_temperature(logits, y):
    logits, y = np.asarray(logits, np.float64), np.asarray(y).astype(int)

    def nll(lt):
        p = softmax(logits, np.exp(lt))
        return -np.mean(np.log(p[np.arange(len(y)), y] + 1e-12))
    r = minimize_scalar(nll, bounds=(np.log(0.05), np.log(20.0)), method="bounded")
    return float(np.exp(r.x))


def _midrank(x):
    return st.rankdata(x)


def delong(y, s1, s2):
    y = np.asarray(y).astype(bool)
    preds = np.vstack([s1, s2]).astype(np.float64)
    if not np.isfinite(preds).all():
        return float("nan"), float("nan"), float("nan")
    pos, neg = preds[:, y], preds[:, ~y]
    m, n = pos.shape[1], neg.shape[1]
    if m == 0 or n == 0:
        return float("nan"), float("nan"), float("nan")
    tx = np.stack([_midrank(r) for r in pos])
    ty = np.stack([_midrank(r) for r in neg])
    tz = np.stack([_midrank(r) for r in np.hstack([pos, neg])])
    auc = tz[:, :m].sum(1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if var <= 0:
        return float(auc[0]), float(auc[1]), 1.0
    z = (auc[0] - auc[1]) / np.sqrt(var)
    return float(auc[0]), float(auc[1]), float(2 * st.norm.sf(abs(z)))


def recalibrated(ref_val, yv, ref_test, yt, target_sens):
    pv = yv >= C.REFERABLE
    cand = np.unique(ref_val[pv])[::-1]
    thr = cand[-1] if len(cand) else 0.5
    for t in cand:
        if (ref_val[pv] >= t).mean() >= target_sens:
            thr = t
            break
    pt = yt >= C.REFERABLE
    pr = ref_test >= thr
    return float(thr), float(pr[pt].mean()), float((~pr[~pt]).mean())


def spec_at_sens(score, y, target):
    pos = y >= C.REFERABLE
    sp = np.sort(score[pos])[::-1]
    k = int(np.ceil(target * len(sp))) - 1
    thr = sp[min(max(k, 0), len(sp) - 1)]
    return float((score[~pos] < thr).mean())


def adjacent_noise(y, rate, rng):
    y = y.copy()
    flip = rng.random(len(y)) < rate
    step = np.where(rng.random(len(y)) < 0.5, -1, 1)
    step = np.where(y == 0, 1, np.where(y == K - 1, -1, step))
    y[flip] = y[flip] + step[flip]
    return y


In [ ]:
%%writefile /kaggle/working/dr-failure-modes/drfm/run.py
import os
import sys
import json
import glob
import time
import shutil
import argparse
import traceback
import subprocess

from . import config as C

COST = {"resnet50": 3, "vit_small_patch16_224": 2, "efficientnet_b0": 2}
DOMCOST = {"full": 4, "high": 2, "low": 2, "thigh": 1, "tlow": 1}


def _job(kind, protocol, arch, domain, lr, seed):
    j = {"kind": kind, "protocol": protocol, "arch": arch, "domain": domain, "lr": lr, "seed": seed}
    j["id"] = f"{kind}-{protocol}-{arch}-{domain}-lr{lr:g}-s{seed}"
    return j


def lr_choice():
    p = os.path.join(C.OUT, "lr_selection.json")
    if not os.path.exists(p):
        raise RuntimeError("Run the grid phase first")
    with open(p) as f:
        return json.load(f)


def jobs(phase):
    out = []
    if phase == "grid":
        for a in C.ARCHS:
            for lr in C.LR_GRID:
                out.append(_job("train", "main", a, "full", lr, C.SEEDS[0]))
    elif phase == "main":
        lr = lr_choice()
        for a in C.ARCHS:
            for s in C.SEEDS:
                for d in ["full", "high", "low"]:
                    out.append(_job("train", "main", a, d, lr[a], s))
                    out.append(_job("train", "v1", a, d, C.PROTOCOLS["v1"]["lr"], s))
            for s in C.TERTILE_SEEDS:
                for d in ["thigh", "tlow"]:
                    out.append(_job("train", "main", a, d, lr[a], s))
        out.sort(key=lambda j: -(COST.get(j["arch"], 2) * DOMCOST[j["domain"]] * (3 if j["protocol"] == "main" else 1)))
    elif phase == "xai":
        lr = lr_choice()
        for a in C.ARCHS:
            for s in C.SEEDS:
                out.append(_job("xai", "main", a, "full", lr[a], s))
    return out


def _done(j):
    if j["kind"] == "xai":
        return os.path.exists(os.path.join(C.RUNS, "xai", j["arch"], f"seed{j['seed']}", "done.json"))
    from .engine import run_dir
    return os.path.exists(os.path.join(run_dir(j), "done.json"))


def worker(phase):
    from .engine import run_train_job
    from .xai import run_xai_job
    for j in jobs(phase):
        if _done(j):
            continue
        claim = os.path.join(C.CLAIMS, j["id"])
        try:
            fd = os.open(claim, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.close(fd)
        except FileExistsError:
            continue
        print(f"== {j['id']}", flush=True)
        try:
            (run_xai_job if j["kind"] == "xai" else run_train_job)(j)
        except Exception:
            traceback.print_exc()
            sys.stdout.flush()
            with open(os.path.join(C.LOGS, "failed.txt"), "a") as f:
                f.write(j["id"] + "\n")


def launch(phase):
    C.ensure_dirs()
    shutil.rmtree(C.CLAIMS, ignore_errors=True)
    os.makedirs(C.CLAIMS, exist_ok=True)
    js = jobs(phase)
    try:
        import torch
        ngpu = max(1, torch.cuda.device_count())
    except Exception:
        ngpu = 1
    print(f"phase={phase} jobs={len(js)} already_done={sum(_done(j) for j in js)} gpus={ngpu}", flush=True)
    procs = []
    for g in range(ngpu):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        log = open(os.path.join(C.LOGS, f"{phase}_gpu{g}.log"), "a")
        procs.append((subprocess.Popen([sys.executable, "-m", "drfm.run", "worker", "--phase", phase], env=env,
                                       stdout=log, stderr=subprocess.STDOUT, cwd=os.path.dirname(os.path.dirname(os.path.abspath(__file__)))), log))
        time.sleep(5)
    t0 = time.time()
    while any(p.poll() is None for p, _ in procs):
        time.sleep(60)
        print(f"  [{(time.time() - t0) / 60:.0f} min] done {sum(_done(j) for j in js)}/{len(js)}", flush=True)
    for _, log in procs:
        log.close()
    missing = [j["id"] for j in js if not _done(j)]
    print(f"phase {phase} finished: {len(js) - len(missing)}/{len(js)} done", flush=True)
    for m in missing:
        print("  NOT DONE:", m)
    if missing:
        for g in range(ngpu):
            lp = os.path.join(C.LOGS, f"{phase}_gpu{g}.log")
            print(f"--- tail {lp}")
            with open(lp) as f:
                print("".join(f.readlines()[-40:]))
    if phase == "grid":
        select_lr()


def select_lr():
    import pandas as pd
    rows = []
    for j in jobs("grid"):
        from .engine import run_dir
        p = os.path.join(run_dir(j), "done.json")
        if os.path.exists(p):
            with open(p) as f:
                d = json.load(f)
            rows.append({"arch": j["arch"], "lr": j["lr"], "best_val_qwk": d["best_val_qwk"], "best_epoch": d["best_epoch"]})
    df = pd.DataFrame(rows)
    os.makedirs(C.TABLES, exist_ok=True)
    df.to_csv(os.path.join(C.TABLES, "table_lr_selection.csv"), index=False)
    choice = {a: float(g.sort_values(["best_val_qwk", "lr"], ascending=[False, True]).iloc[0]["lr"]) for a, g in df.groupby("arch")}
    with open(os.path.join(C.OUT, "lr_selection.json"), "w") as f:
        json.dump(choice, f, indent=1)
    print(df.to_string(index=False))
    print("selected:", choice)


def prefetch():
    import timm
    for a in C.ARCHS:
        timm.create_model(a, pretrained=True, num_classes=C.NUM_CLASSES)
        print("weights ready:", a)


def pack():
    base = os.path.join(C.WORK, "drfm_results")
    shutil.make_archive(base, "zip", C.OUT)
    print("Saved", base + ".zip", f"({os.path.getsize(base + '.zip') / 1e6:.1f} MB)")


def restore(src):
    for sub in ["outputs", "checkpoints"]:
        s = os.path.join(src, sub)
        if os.path.isdir(s):
            shutil.copytree(s, os.path.join(C.WORK, sub), dirs_exist_ok=True)
            print("restored", sub)
    for z in glob.glob(os.path.join(src, "*.zip")):
        shutil.unpack_archive(z, C.OUT)
        print("restored", z)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("stage", choices=["cache", "prefetch", "launch", "worker", "select", "analyze", "pack", "restore", "all"])
    ap.add_argument("--phase", default="main")
    ap.add_argument("--src", default="")
    a = ap.parse_args()
    C.ensure_dirs()
    if a.stage == "cache":
        from . import data as D
        D.build_aptos_cache()
        D.build_external_cache()
    elif a.stage == "prefetch":
        prefetch()
    elif a.stage == "launch":
        launch(a.phase)
    elif a.stage == "worker":
        worker(a.phase)
    elif a.stage == "select":
        select_lr()
    elif a.stage == "analyze":
        from .analyze import run_all
        run_all()
    elif a.stage == "pack":
        pack()
    elif a.stage == "restore":
        restore(a.src)
    elif a.stage == "all":
        from . import data as D
        D.build_aptos_cache()
        D.build_external_cache()
        for ph in ["grid", "main", "xai"]:
            launch(ph)
        from .analyze import run_all
        run_all()
        pack()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/dr-failure-modes/drfm/xai.py
import os
import json

import cv2
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

from . import config as C
from . import data as D
from .corruptions import corrupt
from .engine import DEV, build_model, ckpt_path, prep


class RefLogOdds:
    def __call__(self, out):
        return torch.logsumexp(out[..., C.REFERABLE:], -1) - torch.logsumexp(out[..., :C.REFERABLE], -1)


def _layer(model, arch):
    if arch.startswith("vit"):
        return model.blocks[-1].norm1
    if arch.startswith("resnet"):
        return model.layer4[-1]
    return model.conv_head


def _reshape(model):
    k = getattr(model, "num_prefix_tokens", 1)

    def f(t):
        n = t.size(1) - k
        h = int(round(n ** 0.5))
        return t[:, k:, :].reshape(t.size(0), h, h, t.size(2)).permute(0, 3, 1, 2)
    return f


def load_cam(arch, seed, lr):
    from pytorch_grad_cam import GradCAM
    m = build_model(arch).float()
    sd = torch.load(ckpt_path(arch, seed, lr), map_location=DEV)
    m.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
    m.eval()
    rt = _reshape(m) if arch.startswith("vit") else None
    return m, GradCAM(model=m, target_layers=[_layer(m, arch)], reshape_transform=rt)


def cams(cam, imgs, targets, bs=32):
    out = []
    for i in range(0, len(imgs), bs):
        x = prep(torch.from_numpy(np.ascontiguousarray(imgs[i:i + bs])).permute(0, 3, 1, 2))
        out.append(cam(input_tensor=x, targets=targets[i:i + bs]).astype(np.float32))
    return np.concatenate(out)


def _crop(a):
    return np.ascontiguousarray(D.center_crop(a))


def _conds():
    out = [("clean", 0)]
    for c in C.CORRUPTIONS:
        for s in C.XAI_SEVERITIES:
            out.append((c, s))
    return out


def run_xai_job(job):
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    out_dir = os.path.join(C.RUNS, "xai", job["arch"], f"seed{job['seed']}")
    if os.path.exists(os.path.join(out_dir, "done.json")):
        return
    os.makedirs(out_dir, exist_ok=True)
    meta, arr, splits = D.load_aptos()
    te = splits["full"][2]
    tm = meta.iloc[te].reset_index(drop=True)
    model, cam = load_cam(job["arch"], job["seed"], job["lr"])
    clean = np.stack([_crop(arr[i]) for i in te])
    fov = (clean.mean(-1) / 255.0 > 0.10).astype(np.float32)
    samp = list(tm.sample(n=min(6, len(tm)), random_state=0).index)
    rows, keep = [], {}
    for c, s in _conds():
        imgs = clean if c == "clean" else np.stack([_crop(corrupt(arr[i], c, s, k)) for i, k in zip(te, tm["id_code"])])
        g = cams(cam, imgs, [ClassifierOutputTarget(int(t)) for t in tm["diagnosis"]])
        e = (g * fov).sum((1, 2)) / (g.sum((1, 2)) + 1e-9)
        for j in range(len(tm)):
            rows.append({"dataset": "aptos", "image": tm["id_code"][j], "grade": int(tm["diagnosis"][j]),
                         "corruption": c, "severity": s, "energy_on_retina": float(e[j])})
        if job["seed"] == 0:
            keep[f"aptos/{c}/{s}"] = g[samp].astype(np.float16)
    if job["seed"] == 0:
        keep["aptos/ids"] = np.array(tm["id_code"][samp].tolist())
        keep["aptos/grades"] = tm["diagnosis"].values[samp]
    sm, sa = D.load_external("idrid_seg")
    if sm is not None:
        masks = np.load(os.path.join(C.CACHE, "idrid_seg_masks_256.npy"), mmap_mode="r")
        sclean = np.stack([_crop(a) for a in sa])
        sfov = sclean.mean(-1) / 255.0 > 0.10
        les = np.stack([_crop(m.transpose(1, 2, 0)).transpose(2, 0, 1) for m in masks]).astype(bool)
        union = les.any(1)
        kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
        near = np.stack([cv2.dilate(u.astype(np.uint8), kern) > 0 for u in union])
        for c, s in _conds():
            imgs = sclean if c == "clean" else np.stack([_crop(corrupt(a, c, s, k)) for a, k in zip(sa, sm["image"])])
            g = cams(cam, imgs, [RefLogOdds() for _ in range(len(imgs))])
            for j in range(len(sm)):
                f, u, gj = sfov[j], union[j] & sfov[j], g[j]
                tot = float((gj * f).sum()) + 1e-9
                area = u.sum() / max(1, f.sum())
                frac = float((gj * u).sum()) / tot
                gm = np.where(f, gj, -1)
                yx = np.unravel_index(int(gm.argmax()), gm.shape)
                try:
                    pa = float(roc_auc_score(u[f], gj[f])) if 0 < u[f].sum() < f.sum() else float("nan")
                except Exception:
                    pa = float("nan")
                r = {"dataset": "idrid_seg", "image": sm["image"][j], "grade": -1, "corruption": c, "severity": s,
                     "energy_on_retina": float((gj * f).sum() / (gj.sum() + 1e-9)), "lesion_fraction": frac,
                     "lesion_area": float(area), "lesion_lift": frac / area if area > 0 else float("nan"),
                     "pointing_hit": float(near[j][yx]), "pixel_auc": pa}
                for t, name in enumerate(["MA", "HE", "EX", "SE"]):
                    lt = les[j, t] & f
                    a_t = lt.sum() / max(1, f.sum())
                    r[f"lift_{name}"] = float((gj * lt).sum() / tot / a_t) if a_t > 0 else float("nan")
                rows.append(r)
            if job["seed"] == 0:
                keep[f"idrid/{c}/{s}"] = g[:4].astype(np.float16)
        if job["seed"] == 0:
            keep["idrid/ids"] = np.array(sm["image"][:4].tolist())
    pd.DataFrame(rows).to_csv(os.path.join(out_dir, "xai.csv"), index=False)
    if keep:
        np.savez_compressed(os.path.join(out_dir, "cams.npz"), **keep)
    with open(os.path.join(out_dir, "done.json"), "w") as f:
        json.dump(job, f)
    del model, cam
    if DEV == "cuda":
        torch.cuda.empty_cache()


In [ ]:
import os, subprocess, sys
os.chdir("/kaggle/working/dr-failure-modes")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "grad-cam", "timm"], check=False)
os.environ["DRFM_QUICK"] = "1" if QUICK else "0"
os.environ["DRFM_WORKERS"] = "2"
import torch
print("QUICK =", QUICK, "| torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
for d in sorted(os.listdir("/kaggle/input")):
    print("input:", d, os.listdir(os.path.join("/kaggle/input", d))[:6])

In [ ]:
!python -m drfm.run cache

In [ ]:
!python -m drfm.run prefetch

In [ ]:
!python -m drfm.run launch --phase grid

In [ ]:
!python -m drfm.run launch --phase main

In [ ]:
!python -m drfm.run launch --phase xai

In [ ]:
!python -m drfm.run analyze

In [ ]:
!python -m drfm.run pack